## Clase 3 - Proyecto Práctico 3 – Utilizar Spark Streaming con MLlib.

###### Contexto: Partiremos de un dataset publico con datos biometricos para la clasificacion de ataques al corazon.
###### Dataset: El conjunto de datos consiste de 303 filas y 14 columnas que representan informacion de un paciente. El dataset puede descargarse del siguiente enlace: https://www.kaggle.com/datasets/rashikrahmanpritom/heart-attack-analysis-prediction-dataset
###### Objetivo: A partir del dataset con las observaciones de datos de los pacientes vamos construir un modelo de Regresion Logistica que permita predecir si un paciente tiene mas chances de padecer un ataque al corazon o menos probabilidades.  La columna output nos indica con un valor de 0 si el paciente tiene menos chances de padecer un problema cardiaco o con un valor de 1 si el paciente tiene mas probabilidades de padecerlo.

# 1. Configuración de PySpark en Google Colab

In [1]:
# Instalar Java
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Descargar Spark 3.5.1
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz

# Extraer los archivos de Spark
!tar xf spark-3.5.1-bin-hadoop3.tgz

# Instalar findspark para facilitar el uso de Spark en Python
!pip install -q findspark

# Configurar las variables de entorno para Java y Spark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

# Inicializar Spark usando findspark
import findspark
findspark.init()

# Crear la sesión de Spark
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

# Verificar la versión de Spark
print(f"Versión de Spark: {spark.version}")




Versión de Spark: 3.5.1


# 2. Montamos Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 3. Importamos las librerias requeridas

In [3]:
from pyspark.ml import Pipeline
from pyspark.sql.types import StructType,StructField,LongType, StringType,DoubleType,TimestampType
from pyspark.sql.functions import col

# 4. Creamos un esquema para el DataFrame

In [4]:
schema = StructType( \
                     [StructField("age", LongType(),True), \
                      StructField("sex", LongType(), True), \
                      StructField("cp", LongType(), True), \
                      StructField('trtbps', LongType(), True), \
                      StructField("chol", LongType(), True), \
                      StructField("fbs", LongType(), True), \
                      StructField("restecg", LongType(), True), \
                      StructField("thalachh", LongType(), True),\
                      StructField("exng", LongType(), True), \
                      StructField("oldpeak", DoubleType(), True), \
                      StructField("slp", LongType(),True), \
                      StructField("caa", LongType(), True), \
                      StructField("thall", LongType(), True), \
                      StructField("output", LongType(), True), \
                        ])


# 5. Cargamos el Dataframe desde el archivo .csv y renombramos la columna de salida como "label"

In [ ]:
data_path = '/content/drive/MyDrive/BDPS - Notebooks/Ficheros Input Notebooks/Clase 3/heart.csv'

df = spark.read.format('csv') \
    .option('header', True) \
    .schema(schema) \
    .load(data_path)

# 3. Renombrar la columna "output" a "label"
df = df.withColumnRenamed("output", "label")

# 4. Mostrar el DataFrame (en lugar de df.display(), usamos df.show())
df.show()

# 5. Imprimir el esquema del DataFrame
df.printSchema()

# 6. Creamos los conjuntos de datos de entrenamiento y evaluacion

In [ ]:
testDF, trainDF = df.randomSplit([0.3, 0.7])

#### Cargamos los modulos requeridos para el preprocesamiento del Dataframe

In [ ]:
from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.feature import MinMaxScaler
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.classification import LogisticRegression

#### Realizamos el preprocesamiento del Dataframe

In [ ]:

# Creamos el objeto LogisticRegression
lr = LogisticRegression(maxIter=10, regParam= 0.01)

# Creamos el objeto OneHotEncoder
ohe = OneHotEncoder(inputCols = ['sex', 'cp', 'fbs', 'restecg', 'slp', 'exng', 'caa', 'thall'], outputCols=['sex_ohe', 'cp_ohe', 'fbs_ohe', 'restecg_ohe', 'slp_ohe', 'exng_ohe', 'caa_ohe', 'thall_ohe'])

# Escalan las entradas y creamos las representaciones vectoriales para el modelo
assembler1 = VectorAssembler(inputCols=['age','trtbps','chol','thalachh','oldpeak'], outputCol="features_scaled1")
scaler = MinMaxScaler(inputCol="features_scaled1", outputCol="features_scaled")
assembler2 = VectorAssembler(inputCols=['sex_ohe', 'cp_ohe', 'fbs_ohe', 'restecg_ohe', 'slp_ohe', 'exng_ohe', 'caa_ohe', 'thall_ohe','features_scaled'], outputCol="features")

#Creamos el objecto pipeline
pipeline = Pipeline(stages= [assembler1, scaler, ohe, assembler2,lr])
# Entrenamos el modelo con el conjunto de entrenamiento
pModel = pipeline.fit(trainDF)

# Tranformamos los datos y revisamos los valores para las predicciones
trainingPred = pModel.transform(trainDF)
trainingPred.select('label','probability','prediction').show(20, truncate=False)

# 7. Partimos el archivo .csv original en 10 partes para simular el efecto de streaming.  Borramos el directorio de origen de los datos y luego salvamos las partes del archivo .csv.

In [ ]:
# 1. Definir la ruta en Google Drive donde se guardarán los archivos
output_path = '/content/drive/MyDrive/BDPS - Notebooks/Ficheros Input Notebooks/Clase 3/streaming/'

# 2. Reparticionar los datos (si es necesario)
testData = testDF.repartition(10)

# 3. Eliminar el directorio si ya existe (equivalente a dbutils.fs.rm)
import shutil
import os

# Comprobar si la carpeta existe y eliminarla
if os.path.exists(output_path):
    shutil.rmtree(output_path)  # Eliminar carpeta y su contenido

# 4. Guardar los datos en formato CSV en la ruta de Google Drive
testData.write.format("csv").option("header", True).save(output_path)


# 8. Creamos la fuente de datos del stream

In [ ]:
# 1. Definir la ruta del directorio de streaming en Google Drive
streaming_path = '/content/drive/MyDrive/BDPS - Notebooks/Ficheros Input Notebooks/Clase 3/streaming/'

# 2. Leer el archivo CSV como fuente de streaming
sourceStream = spark.readStream.format("csv") \
    .option("header", True) \
    .schema(schema) \
    .option("ignoreLeadingWhiteSpace", True) \
    .option("mode", "dropMalformed") \
    .option("maxFilesPerTrigger", 1) \
    .load(streaming_path) \
    .withColumnRenamed("output", "label")



#### A partir del modelo creado aplicamos el metodo transform sobre el ReadStream de fuente, seleccionando las columnas que requerimos para verificar los resultados del modelo.

In [ ]:
streamingHeart = pModel.transform(sourceStream).select('label', 'probability', 'prediction')
display(streamingHeart)

#### Creamos una segunda consulta sobre el stream para hacer consultas simultaneas

In [ ]:
query = (streamingHeart.writeStream.outputMode("append").format("memory").queryName("HeartClassification").start())

In [ ]:
# Consulta los datos que se están acumulando en memoria con SQL
result_df = spark.sql("SELECT * FROM HeartClassification")

# Mostrar el resultado
result_df.show()


In [ ]:
HeartClassification_df = spark.sql("select * from HeartClassification")

# 9. Ahora, vamos a ver en el DF cuando se ha equivocado el modelo comparando label (es decir, los casos reales), con la prediction (es decir, lo que hemos obtenido como resultado del modelo)

In [ ]:
HeartClassification_df.select(col("prediction"), col("label")).where("label != prediction").show(30, truncate=False)

# 10. Listamos los Streams activos y los detenemos.

In [ ]:
for stream in spark.streams.active:
  stream.stop()